# Part 5A — Hybrid RAG (Notebook 05)

This notebook implements Hybrid RAG on top of the existing 4,000-paper corpus.


## Tutorial Goals

This notebook is a standalone, zero-to-hero tutorial with:

1. Concept explanation from first principles
2. Architecture and workflow breakdown
3. End-to-end implementation code
4. Real execution outputs and benchmark metrics
5. Practical analysis and production takeaways


## What is this technique?

        ### Definition and core concepts
        Hybrid RAG combines dense semantic retrieval and sparse lexical retrieval to improve robustness.

        ### Why was this technique developed?
        Single retrievers fail on mixed query types: dense misses exact terms and BM25 misses semantic paraphrases.

        ### What limitations of traditional RAG does it solve?
        It addresses acronym-heavy misses, vocabulary mismatch, and incomplete context recall.

        ### Architecture and workflow diagram explanation

```mermaid
graph TD
    Q[Query] --> D[Dense Retrieval]
    Q --> B[BM25 Retrieval]
    D --> F[Fusion]
    B --> F
    F --> C[Context]
    C --> G[Generation]
```


        ### Component-by-component breakdown
        Dense retriever, BM25 retriever, fusion strategy, generation stage, evaluation metrics.

        ### When should it be used in real-world systems?
        Use in technical corpora and enterprise search where both semantic and exact term matching are important.

        ### Advantages and disadvantages
        **Advantages**
        - Better overall retrieval coverage
- Stronger robustness on heterogeneous queries
- Practical production baseline

        **Disadvantages**
        - More components than naive RAG
- Requires fusion tuning
- Slight latency overhead

        ### Comparison against standard RAG and other implemented RAG variants
        Compared with standard RAG, Hybrid RAG is more stable. Compared with GraphRAG, it is simpler but less relation-aware.

        ### Implementation details and design decisions used in this project
        This implementation reuses the existing FAISS index and adds BM25 over the same chunks, then benchmarks Dense vs BM25 vs Hybrid.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_v2.data import load_base_corpus, load_papers_from_chunks
from src.rag_v2.retrieval import DenseRetriever, BM25Retriever, HybridRetriever
from src.rag_v2.metrics import build_keyword_eval_set, compute_retrieval_metrics, save_json

ART = PROJECT_ROOT / 'artifacts' / 'rag_v2'
ART.mkdir(parents=True, exist_ok=True)

index, chunks = load_base_corpus()
papers = load_papers_from_chunks(chunks)

display(Markdown(f"Loaded **{len(chunks):,} chunks** from **{len(papers):,} papers** (FAISS dim={index.d})."))


In [ ]:
dense = DenseRetriever(index=index, chunks=chunks)
bm25 = BM25Retriever(chunks=chunks)
hybrid = HybridRetriever(dense=dense, bm25=bm25, alpha=0.7)

eval_set = build_keyword_eval_set(papers)
results = compute_retrieval_metrics(eval_set, {'Dense': dense, 'BM25': bm25, 'Hybrid': hybrid}, k=5)

hybrid_df = pd.DataFrame(results).T
hybrid_df


In [ ]:
import ollama
from src.rag_v2.agentic import llm_faithfulness

sample_questions = [
    'How does RLHF work?',
    'Why is LoRA parameter-efficient?',
    'What is speculative decoding?',
]

rows = []
for q in sample_questions:
    docs = hybrid.retrieve(q, k=6)
    context = "\n\n".join(d['text'][:900] for d in docs)
    prompt = f"Use only context.\n\nContext:\n{context}\n\nQuestion: {q}\nAnswer:"
    t0 = time.perf_counter()
    resp = ollama.chat(
        model='granite4.1:8b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.2, 'num_gpu': 0},
    )
    latency = (time.perf_counter() - t0) * 1000
    answer = resp['message']['content'].strip()
    faith, reason = llm_faithfulness(
        q,
        answer,
        [d['text'] for d in docs],
        judge_model='granite4.1:8b',
    )
    rows.append(
        {
            'question': q,
            'faithfulness': round(faith, 3),
            'latency_ms': round(latency, 2),
            'reason': reason,
        }
    )

gen_df = pd.DataFrame(rows)
gen_df

In [ ]:
out_json = ART / 'hybrid' / '05_hybrid_metrics.json'
out_json.parent.mkdir(parents=True, exist_ok=True)
save_json(out_json, {'retrieval': results, 'generation': gen_df.to_dict(orient='records')})
out_json


In [ ]:
best_retriever = hybrid_df['recall@5'].astype(float).idxmax()
mean_faith = float(gen_df['faithfulness'].mean())

if float(hybrid_df.loc['Hybrid', 'recall@5']) == 0.0 and float(hybrid_df.loc['BM25', 'recall@5']) == 0.0:
    retrieval_note = 'All weak-supervision anchors were missed in this run, indicating evaluation-keyword mismatch rather than a clear retriever winner.'
else:
    retrieval_note = f'The best Recall@5 in this run was from **{best_retriever}**.'

if mean_faith < 0.6:
    faith_note = 'Generation faithfulness is limited because retrieved context for sampled questions was often insufficient.'
else:
    faith_note = 'Generation faithfulness is reasonably stable for sampled questions.'

analysis = (
    "## Post-run Analysis (Real Results)\n\n"
    f"- Dense Recall@5: **{hybrid_df.loc['Dense','recall@5']:.4f}**\n"
    f"- BM25 Recall@5: **{hybrid_df.loc['BM25','recall@5']:.4f}**\n"
    f"- Hybrid Recall@5: **{hybrid_df.loc['Hybrid','recall@5']:.4f}**\n"
    f"- Hybrid P50 latency: **{hybrid_df.loc['Hybrid','latency_p50_ms']:.2f} ms**\n"
    f"- Hybrid P95 latency: **{hybrid_df.loc['Hybrid','latency_p95_ms']:.2f} ms**\n"
    f"- Mean faithfulness: **{mean_faith:.3f}**\n\n"
    f"- Observation: {retrieval_note}\n"
    f"- Observation: {faith_note}"
)
display(Markdown(analysis))
